Given a question about a past decision, retrieve the relevant architecture decision record (ADR).

Engineering teams accumulate institutional knowledge in ADRs, RFCs, and design docs. An AI agent with episodic memory can answer 'why did we choose X?' by retrieving the relevant record.

**Corpus:** [Neon](https://neon.tech) architecture decision records — 247 documents covering storage, consensus, page format, control plane, and deployment decisions.

**Challenge:** Questions use current terminology while ADRs use the language of the time they were written. Temporal and domain-specific vocabulary drift is the key difficulty.

In [ ]:
import contextlib, json, pathlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BENCHMARK = 'episodic-memory'
ROOT = pathlib.Path().resolve()
for _p in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (_p / 'results').exists():
        ROOT = _p; break
RESULTS_DIR = ROOT / 'results'

ADAPTERS = ['sqlite', 'lancedb', 'chromadb', 'tantivy', 'qdrant']
ADAPTER_LABELS = {
    'sqlite': 'SQLite FTS5', 'lancedb': 'LanceDB',
    'chromadb': 'ChromaDB', 'tantivy': 'Tantivy', 'qdrant': 'Qdrant'
}
_OUTER_BG = '#f5f3ef'; _PLOT_BG = '#ffffff'; _MUTED = '#6b6b6b'
_SPINE = '#d8d5d0'; _GRID = '#ebebeb'; _LABEL_CLR = '#7a7370'; _INK = '#1a1917'
_ADAPTER_COLORS = {'sqlite': '#bdb9b5', 'lancedb': '#3d8c7a', 'chromadb': '#4b7ebb', 'tantivy': '#d4952a', 'qdrant': '#edc948'}
_FALLBACK = ['#c96442', '#4b7ebb', '#3d8c7a', '#d4952a', '#bdb9b5']
_P50 = '#e8903a'; _P95 = '#7eb8d4'
_TS = 9; _LS = 8; _TIS = 11.5
mpl.rcParams.update({'figure.dpi': 96, 'font.family': 'sans-serif', 'font.size': _TS,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': _GRID, 'grid.linewidth': 0.7, 'grid.linestyle': '-', 'axes.axisbelow': True})

def _sty(fig, ax):
    fig.patch.set_facecolor(_OUTER_BG); ax.set_facecolor(_PLOT_BG)
    for s in ['left','bottom']: ax.spines[s].set_color(_SPINE); ax.spines[s].set_linewidth(0.7)
    ax.tick_params(axis='both', colors=_MUTED, labelsize=_TS, length=3, width=0.7)
    ax.xaxis.label.set_color(_MUTED); ax.yaxis.label.set_color(_MUTED)

def _bc(s, i): return _ADAPTER_COLORS.get(s.lower(), _FALLBACK[i % len(_FALLBACK)])

rows = []
for f in RESULTS_DIR.glob('**/*.json'):
    with contextlib.suppress(Exception): rows.append(json.loads(f.read_text()))
df = pd.DataFrame(rows) if rows else pd.DataFrame()
bdf = (df[df['benchmark'] == BENCHMARK]
       .sort_values('ndcg_at_10', ascending=False)
       .groupby('store').first()
       .reindex(ADAPTERS))
print(f'Results for {BENCHMARK}: {len(bdf.dropna(subset=["ndcg_at_10"])) if not bdf.empty else 0} adapters')

## Data and Search Overview

### ADR Retrieval Flow

```mermaid
flowchart LR
    Q["Team question\n(e.g. 'Why did we\nchoose Paxos\nover Raft?')"] --> R["ADR search\n(query 247 records)"]
    R --> ADR["Matched ADR\n(e.g. ADR-017:\nConsensus Protocol)"]
    ADR --> A["Structured answer:\nContext \u00b7 Decision\n\u00b7 Consequences \u00b7 Status"]
    style Q fill:#e3f2fd,stroke:#1565c0
    style A fill:#e8f5e9,stroke:#2e7d32
```


In [ ]:
import json, pathlib
import matplotlib.pyplot as plt
import numpy as np

ROOT = pathlib.Path().resolve()
for _p in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (_p / 'results').exists(): ROOT = _p; break

result_dir = ROOT / 'results' / 'episodic-memory' / 'tantivy'
files = sorted(result_dir.glob('*.json')) if result_dir.exists() else []
meta = json.loads(files[-1].read_text()) if files else {}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
fig.patch.set_facecolor('#fafafa')

# Left: ADR topic distribution (approximate — based on Neon ADR repository structure)
ax = axes[0]; ax.set_facecolor('#fafafa')
for s in ax.spines.values(): s.set_visible(False)
topics = ['Storage / WAL', 'Consensus\n(Paxos/Raft)', 'Page format', 'Control plane', 'Deployment', 'Other']
counts = [62, 48, 41, 38, 33, 25]
colors_t = ['#4e79a7', '#e15759', '#59a14f', '#f28e2b', '#76b7b2', '#b07aa1']
wedges, texts, pcts = ax.pie(counts, labels=topics, colors=colors_t, autopct='%1.0f%%',
                              startangle=90, pctdistance=0.75, textprops={'fontsize': 8.5})
ax.set_title(f'Neon ADR corpus: {meta.get("num_docs", 247)} records by topic (approx.)', fontsize=9)

# Right: nDCG@10 — BM25 adapters lead
ax2 = axes[1]; ax2.set_facecolor('#fafafa')
for s in ax2.spines.values(): s.set_visible(False)
adapters_ord = ['sqlite', 'tantivy', 'lancedb', 'chromadb']
labels_m = {'sqlite': 'SQLite\nFTS5\n(BM25)', 'tantivy': 'Tantivy\n(BM25)', 'lancedb': 'LanceDB\n(dense)', 'chromadb': 'ChromaDB\n(dense)'}
cols_m = {'sqlite': '#f28e2b', 'tantivy': '#e15759', 'lancedb': '#4e79a7', 'chromadb': '#59a14f'}
scores = {}
for a in adapters_ord:
    d = ROOT / 'results' / 'episodic-memory' / a
    fs = sorted(d.glob('*.json')) if d.exists() else []
    if fs: scores[a] = json.loads(fs[-1].read_text()).get('ndcg_at_10', 0)
vals = [scores.get(a, 0) for a in adapters_ord]
bars = ax2.bar([labels_m[a] for a in adapters_ord], vals, color=[cols_m[a] for a in adapters_ord], width=0.5, zorder=3)
for b, v in zip(bars, vals):
    ax2.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
ax2.set_ylabel('nDCG@10'); ax2.set_ylim(0, 0.9)
ax2.set_title('BM25 leads: ADRs use exact engineering vocabulary', fontsize=9)
ax2.yaxis.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout(); plt.show()


## Background

### What This Benchmark Measures

Given a question about a past engineering decision (e.g., "Why did we choose Paxos over Raft for the safekeeper consensus protocol?"), retrieve the Architecture Decision Record (ADR) that answers it.

ADRs are a standard practice in software engineering: short documents (typically 200–800 words) that record a decision, its context, and its consequences. An AI agent with episodic memory uses ADR retrieval to answer "why did we do X?" questions — recalling institutional knowledge that doesn't belong in code comments.

**Corpus:** 247 ADRs from the [Neon](https://neon.tech) open-source PostgreSQL-compatible cloud database. Neon publishes its architecture decision records openly; topics include storage engine design, consensus protocols, page format, and control plane architecture.

**Ground truth:** Each query is a question derived from the ADR content. The paired ADR is the gold relevant document. All other ADRs are non-relevant.

**Queries per ADR:** ~4.5 on average (1,114 queries for 247 ADRs via BEIR-style multi-query evaluation).

### Why BM25 Wins Here

Neon ADRs use highly specific technical vocabulary: "Paxos", "WAL", "safekeeper", "HNSW", "page server", "compute node". These terms are domain-specific jargon that users include verbatim in their questions. When query and document share this precise terminology, BM25's inverted-index lookup with IDF weighting assigns high scores to exact matches — no semantic bridging needed.

Dense search must still *understand* "safekeeper replication protocol" maps to "consensus mechanism for write-ahead log propagation" — and it does reasonably well (LanceDB: 0.640). But BM25 simply wins when the vocabulary is unambiguous and shared.

This result suggests: for highly domain-specific corpora with stable, precise vocabulary, keyword search is hard to beat.

### References

1. [Neon ADR repository (open-source)](https://github.com/neondatabase/neon/tree/main/docs/rfcs)
2. Michael Nygard (2011). *Documenting Architecture Decisions.* [cognitect.com/blog](https://cognitect.com/blog/2011/11/15/documenting-architecture-decisions)
3. Robertson, S. & Zaragoza, H. (2009). The probabilistic relevance framework: BM25 and beyond. [doi:10.1561/1500000019](https://doi.org/10.1561/1500000019)
4. Thakur, N. et al. (2021). BEIR: A heterogeneous benchmark for zero-shot evaluation of information retrieval models. [arXiv:2104.08663](https://arxiv.org/abs/2104.08663)
5. Lewis, P. et al. (2020). *Retrieval-augmented generation for knowledge-intensive NLP tasks.* NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401) *(foundational RAG paper)*


## Results

In [ ]:
cols = ['Adapter', 'nDCG@10', 'R@1', 'R@5', 'R@10', 'MRR@10', 'p50 (ms)']
rows_t = []
for a in ADAPTERS:
    if bdf.empty or a not in bdf.index or pd.isna(bdf.loc[a].get('ndcg_at_10')): continue
    r = bdf.loc[a]
    rows_t.append({'Adapter': ADAPTER_LABELS[a], 'nDCG@10': f"{r.get('ndcg_at_10',0):.3f}",
        'R@1': f"{r.get('recall_at_1',0):.3f}", 'R@5': f"{r.get('recall_at_5',0):.3f}",
        'R@10': f"{r.get('recall_at_10',0):.3f}", 'MRR@10': f"{r.get('mrr_at_10',0):.3f}",
        'p50 (ms)': f"{r.get('latency_p50_ms',0):.2f}"})
if rows_t:
    from IPython.display import display, HTML
    tdf = pd.DataFrame(rows_t, columns=cols)
    display(HTML(tdf.to_html(index=False, classes='results-table', border=0)))
else:
    from IPython.display import display, HTML
    display(HTML('<p><em>No results.</em></p>'))

In [ ]:
valid = [(a, bdf.loc[a,'ndcg_at_10']) for a in ADAPTERS if not bdf.empty and a in bdf.index and not pd.isna(bdf.loc[a,'ndcg_at_10'])]
if valid:
    stores, vals = zip(*valid)
    labels = [ADAPTER_LABELS.get(s,s) for s in stores]
    colors = [_bc(s,i) for i,s in enumerate(stores)]
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    _sty(fig, ax)
    bars = ax.bar(labels, vals, color=colors, width=0.5, zorder=3)
    ax.set_ylabel('nDCG@10'); ax.set_ylim(0, 1.1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.015, f'{val:.3f}',
                ha='center', va='bottom', fontsize=_LS, color=_LABEL_CLR)
    ax.set_title('nDCG@10 by adapter', color=_INK, fontsize=_TIS, fontweight='bold', pad=10)
    plt.tight_layout(pad=1.0); plt.show()

In [ ]:
valid_lat = [(a, bdf.loc[a,'latency_p50_ms'], bdf.loc[a,'latency_p95_ms'])
             for a in ADAPTERS if not bdf.empty and a in bdf.index and not pd.isna(bdf.loc[a].get('latency_p50_ms'))]
fig, ax = plt.subplots(figsize=(5.5, 3.2))
_sty(fig, ax)
if valid_lat:
    stores, p50, p95 = zip(*valid_lat)
    labels = [ADAPTER_LABELS.get(s,s) for s in stores]
    x = np.arange(len(labels)); w = 0.3
    ax.bar(x-w/2, p50, w, label='p50', color=_P50, zorder=3)
    ax.bar(x+w/2, p95, w, label='p95', color=_P95, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('ms')
    ax.legend(fontsize=_LS, framealpha=0, labelcolor=_MUTED, handlelength=1.0)
else:
    ax.text(0.5, 0.5, 'No latency data', ha='center', va='center', color=_MUTED, transform=ax.transAxes)
ax.set_title('Query latency (ms)', color=_INK, fontsize=_TIS, fontweight='bold', pad=10)
plt.tight_layout(pad=1.0); plt.show()

## Analysis

All four adapters perform well, with Tantivy (0.700) and SQLite FTS5 (0.695) slightly ahead of LanceDB (0.640) and ChromaDB (0.528). The Neon ADRs use precise engineering terminology (Paxos, WAL, safekeeper, HNSW) that maps directly to query terms, giving BM25 a competitive edge.

The overall high scores (≥0.528) suggest that Neon ADRs are relatively easy to retrieve — each ADR covers a distinct topic with unique vocabulary. Real-world episodic memory corpora with overlapping or evolving terminology would be harder.

## Limitations

- **Single organisation:** Neon's ADR corpus may not generalise to other organisations' writing styles or decision formats.
- **Short documents:** ADRs are concise; longer, more ambiguous documents would be harder.
- **No temporal context:** queries do not include a date, so time-sensitive retrievals (e.g., `before the 2023 refactor`) are not evaluated.